In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/Tracefinder"

print("Exists:", os.path.exists(BASE_PATH))
print("Contents:", os.listdir(BASE_PATH) if os.path.exists(BASE_PATH) else "Not found")


In [ ]:
import os
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import numpy as np
import pandas as pd
from tqdm import tqdm
from skimage.filters import gaussian

# ================= PATHS =================
BASE_ROOT = "/content/drive/MyDrive/Tracefinder"
RAW_SOURCES = ["Wikipedia", "Official"]

OUT_IMG = os.path.join(BASE_ROOT, "processed/images")
OUT_NOISE = os.path.join(BASE_ROOT, "processed/noise_maps")
META_DIR = os.path.join(BASE_ROOT, "processed/metadata")

os.makedirs(OUT_IMG, exist_ok=True)
os.makedirs(OUT_NOISE, exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)

TARGET_SIZE = (512, 512)
EXTS = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")

# ---------------- UTIL ----------------
def preprocess_image(im):
    im = im.convert("L")
    im = im.resize(TARGET_SIZE)
    return np.asarray(im).astype(np.float32) / 255.0

def extract_noise(arr):
    return arr - gaussian(arr, sigma=1)

def save_arr(arr, path):
    mn, mx = arr.min(), arr.max()
    arr = (arr - mn) / (mx - mn + 1e-8)
    Image.fromarray((arr * 255).astype("uint8")).save(path)

# ---------------- MAIN ----------------
rows = []

for source in RAW_SOURCES:
    src_root = os.path.join(BASE_ROOT, source)

    for scanner in os.listdir(src_root):
        scanner_dir = os.path.join(src_root, scanner)
        if not os.path.isdir(scanner_dir):
            continue

        for dpi in ["150", "300"]:
            dpi_dir = os.path.join(scanner_dir, dpi)
            if not os.path.exists(dpi_dir):
                continue

            files = [f for f in os.listdir(dpi_dir) if f.lower().endswith(EXTS)]

            for f in tqdm(files, desc=f"{source}-{scanner}-{dpi}"):
                img_path = os.path.join(dpi_dir, f)

                try:
                    im = Image.open(img_path)
                    arr = preprocess_image(im)
                    noise = extract_noise(arr)

                    base = f"{source}_{scanner}_{dpi}_{os.path.splitext(f)[0]}"
                    out_img = f"{OUT_IMG}/{base}.png"
                    out_noise = f"{OUT_NOISE}/{base}_noise.png"

                    save_arr(arr, out_img)
                    save_arr(noise, out_noise)

                    rows.append([
                        source, scanner, dpi,
                        img_path, out_img, out_noise
                    ])

                except Exception as e:
                    print("ERROR:", img_path, e)

# Save labels
labels_df = pd.DataFrame(
    rows,
    columns=["source", "scanner", "dpi", "original", "processed", "noise"]
)

labels_df.to_csv(f"{META_DIR}/dataset_labels.csv", index=False)

print("✅ Preprocessing complete")
print("Rows:", len(labels_df))


In [ ]:
import os

img_dir = "/content/drive/MyDrive/Tracefinder/processed/images"
noise_dir = "/content/drive/MyDrive/Tracefinder/processed/noise_maps"

print("Processed images:", len(os.listdir(img_dir)))
print("Noise maps:", len(os.listdir(noise_dir)))

print("\nSample processed images:")
print(os.listdir(img_dir)[:5])

print("\nSample noise maps:")
print(os.listdir(noise_dir)[:5])


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from skimage.feature import local_binary_pattern
from scipy.stats import skew, kurtosis

# ================= PATHS =================
BASE_ROOT = "/content/drive/MyDrive/Tracefinder"
LABELS_CSV = f"{BASE_ROOT}/processed/metadata/dataset_labels.csv"
OUT_DIR = f"{BASE_ROOT}/features"
OUT_CSV = f"{OUT_DIR}/features.csv"

os.makedirs(OUT_DIR, exist_ok=True)

LBP_RADII = [1, 2, 3]
LBP_POINTS = 8
FFT_BINS = 5

# -------- FFT --------
def fft_radial_energy(img, bins=FFT_BINS):
    f = np.fft.fftshift(np.fft.fft2(img))
    mag = np.abs(f)

    h, w = mag.shape
    cy, cx = h // 2, w // 2
    y, x = np.indices((h, w))
    r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
    max_r = r.max()

    return [
        np.mean(mag[(r >= i*max_r/bins) & (r < (i+1)*max_r/bins)])
        for i in range(bins)
    ]

# -------- FEATURES --------
def extract_features(noise_path):
    img = cv2.imread(noise_path, cv2.IMREAD_GRAYSCALE)
    img = img.astype(np.float32) / 255.0

    feats = [
        img.mean(),
        img.std(),
        skew(img.flatten()),
        kurtosis(img.flatten()),
        np.mean(img**2)
    ]

    feats += fft_radial_energy(img)

    for r in LBP_RADII:
        lbp = local_binary_pattern(img, LBP_POINTS*r, r, method="uniform")
        hist, _ = np.histogram(
            lbp,
            bins=np.arange(0, LBP_POINTS*r + 3),
            density=True
        )
        feats.extend(hist)

    return feats

# -------- MAIN --------
df = pd.read_csv(LABELS_CSV)
rows = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    feats = extract_features(row["noise"])
    rows.append([row["scanner"]] + feats)

columns = (
    ["scanner",
     "noise_mean","noise_std","noise_skew","noise_kurt","noise_energy"]
    + [f"fft_{i}" for i in range(FFT_BINS)]
    + [f"lbp_{i}" for i in range(len(rows[0]) - 1 - 5 - FFT_BINS)]
)

pd.DataFrame(rows, columns=columns).to_csv(OUT_CSV, index=False)
print("✅ Features saved:", OUT_CSV)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

FEATURES_CSV = "/content/drive/MyDrive/Tracefinder/features/features.csv"

df = pd.read_csv(FEATURES_CSV)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

X = df.drop("scanner", axis=1)
y = df["scanner"]

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

svm = SVC(kernel="rbf", C=10, gamma="scale")
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, rf_pred))
print("SVM Accuracy:", accuracy_score(y_test, svm_pred))

print("\nRF Classification Report:\n",
      classification_report(y_test, rf_pred))


patch extraction


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

BASE_PATH = "/content/drive/MyDrive/Tracefinder"
LABELS_CSV = f"{BASE_PATH}/processed/metadata/dataset_labels.csv"
PATCH_DIR = f"{BASE_PATH}/patches"
os.makedirs(PATCH_DIR, exist_ok=True)

PATCH_SIZE = 128
STRIDE = 128

df = pd.read_csv(LABELS_CSV)

patch_rows = []

def extract_patches(img, patch_size=128, stride=128):
    h, w = img.shape
    patches = []
    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            patches.append(img[y:y+patch_size, x:x+patch_size])
    return patches

for _, row in tqdm(df.iterrows(), total=len(df)):
    noise_path = row["noise"]
    img = cv2.imread(noise_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue

    img = img.astype(np.float32) / 255.0
    patches = extract_patches(img)

    base = os.path.splitext(os.path.basename(noise_path))[0]

    for i, patch in enumerate(patches):
        patch_name = f"{base}_p{i}.npy"
        patch_path = os.path.join(PATCH_DIR, patch_name)
        np.save(patch_path, patch)

        patch_rows.append([
            row["scanner"],
            patch_path,
            row["original"]
        ])

patch_df = pd.DataFrame(
    patch_rows,
    columns=["scanner", "patch_path", "image_path"]
)

PATCH_CSV = f"{BASE_PATH}/patches/patch_labels.csv"
patch_df.to_csv(PATCH_CSV, index=False)

print("✅ Patch extraction done")
print("Total patches:", len(patch_df))


In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.stats import skew, kurtosis
from skimage.feature import local_binary_pattern

PATCH_CSV = f"{BASE_PATH}/patches/patch_labels.csv"
OUT_FEATURES = f"{BASE_PATH}/patches/patch_features.csv"

LBP_POINTS = 8
LBP_RADII = [1, 2]
FFT_BINS = 3

def fft_energy(img, bins=FFT_BINS):
    f = np.fft.fftshift(np.fft.fft2(img))
    mag = np.abs(f)
    h, w = mag.shape
    cy, cx = h//2, w//2
    y, x = np.ogrid[:h, :w]
    r = np.sqrt((x-cx)**2 + (y-cy)**2)
    r = r / r.max()

    feats = []
    for i in range(bins):
        mask = (r >= i/bins) & (r < (i+1)/bins)
        feats.append(np.mean(mag[mask]))
    return feats

df = pd.read_csv(PATCH_CSV)
rows = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    patch = np.load(row["patch_path"])
    px = patch.flatten()

    feats = [
        np.mean(px),
        np.std(px),
        skew(px),
        kurtosis(px),
        np.mean(px**2)
    ]

    feats += fft_energy(patch)

    for r in LBP_RADII:
        lbp = local_binary_pattern(patch, LBP_POINTS*r, r, method="uniform")
        hist, _ = np.histogram(
            lbp,
            bins=np.arange(0, LBP_POINTS*r + 3),
            density=True
        )
        feats.extend(hist)

    rows.append([row["scanner"], row["image_path"]] + feats)

columns = (
    ["scanner", "image_path",
     "mean","std","skew","kurt","energy"]
    + [f"fft_{i}" for i in range(FFT_BINS)]
    + [f"lbp_{i}" for i in range(len(rows[0]) - 2 - 5 - FFT_BINS)]
)

pd.DataFrame(rows, columns=columns).to_csv(OUT_FEATURES, index=False)

print("✅ Patch features saved:", OUT_FEATURES)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

df = pd.read_csv(OUT_FEATURES)

X = df.drop(columns=["scanner","image_path"])
y = df["scanner"]

scaler = StandardScaler()
X = scaler.fit_transform(X)

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

rf = RandomForestClassifier(
    n_estimators=300,
    n_jobs=-1,
    class_weight="balanced",
    random_state=42
)

rf.fit(Xtr, ytr)
yp = rf.predict(Xte)

print("Patch-level accuracy:", accuracy_score(yte, yp))


In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# ================= PATH =================
BASE_PATH = "/content/drive/MyDrive/Tracefinder"
FEATURES_CSV = f"{BASE_PATH}/patches/patch_features.csv"

# ================= LOAD =================
df = pd.read_csv(FEATURES_CSV)
print("Total patches:", len(df))

X = df.drop(columns=["scanner", "image_path"])
y = df["scanner"]

# ================= SPLIT (KEEP INDICES) =================
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# ================= SCALE =================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ================= TRAIN RF =================
rf = RandomForestClassifier(
    n_estimators=300,
    n_jobs=-1,
    class_weight="balanced",
    random_state=42
)

rf.fit(X_train, y_train)
y_pred_patch = rf.predict(X_test)

print("\nPatch-level accuracy:",
      accuracy_score(y_test, y_pred_patch))

# ================= IMAGE-LEVEL VOTING =================
df_test = df.loc[idx_test].copy()
df_test["pred"] = y_pred_patch

final_preds = []

for img, group in df_test.groupby("image_path"):
    true_label = group["scanner"].iloc[0]
    voted_label = Counter(group["pred"]).most_common(1)[0][0]
    final_preds.append((true_label, voted_label))

final_df = pd.DataFrame(final_preds, columns=["true", "pred"])

image_acc = (final_df["true"] == final_df["pred"]).mean()

print("\n🔥 IMAGE-LEVEL ACCURACY AFTER VOTING:", image_acc)

print("\nImage-level Classification Report:\n")
print(classification_report(final_df["true"], final_df["pred"]))
